# Atomic structure analysis with radial (azimuthal) mean & variance profile data
### Jinseok Ryu (jinseuk56@gmail.com)
Please visit the following link for details of data pre-processing ([ePSIC User-Notebooks](https://github.com/jinseuk56/User-Notebooks/tree/master/ePSIC_Standard_Notebooks))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from scipy.signal import find_peaks
import hyperspy.api as hs

import sys
sys.path.append('') # if SENDePSIC isn't installed

# Import from the installed SENDePSIC package
from sendepsic import radial_profile_analysis

In [ ]:
# load data
base_dir = '../Merlin'
subfolders = [''] # subfolder names you want to load and compare, e.g., ['sub1', 'sub2']
final_dir = None # (optional) folder name where the data is stored in

profile_length = 360 # limit the profile size
num_load = 4 # limit the number of data for every subfolder (select files randomly)

include_key = [] # keyword (datetime) for screening (to only include the specified data)
exclude_key = [] # keyword (datetime) for screening (to exclude poor quality data)

use_gpu = False # for potential work in the future # useless at the moment

# The path of EDX data should be like this: ~/EDX/*.rpl
run_analysis = radial_profile_analysis(base_dir, subfolders, 
                                       profile_length, num_load, final_dir,
                                       include_key, exclude_key,
                                       use_gpu=use_gpu,
                                       simult_edx=False, verbose=False)

In [ ]:
run_analysis.set_figure_save_path(path=None, global_save=False)

In [ ]:
plt.close("all")
# Transformation quality check (center beam alignment)
# If there are any data of poor quality, you can exclude them in the cell above (using 'exclude_key')
# crop=[top, bottom, left, right] -> img[top:bottom, left:right] (optional)
# "visual_title=True" will show the datetimes of each data
# "title_font_size" -> change the font size of the titles
run_analysis.center_beam_alignment_check(crop=[0, -1, 0, -1], 
                                         visual_title=True,
                                         title_font_size=10)

In [ ]:
plt.close("all")
# Intensity integration image (BF + DF)
# "visual_title=True" will show the datetimes of each data
# "title_font_size" -> change the font size of the titles
run_analysis.intensity_integration_image(visual_title=True,
                                         title_font_size=10)

In [ ]:
# To simulate diffraction patterns (XRD) from prismatic structure file(s) (optional)
str_path = [] # structure paths to compare, e.g., ['path1', 'path2']

# Specify the scattering vector range -> also used in NMF decomposition and plotting
from_unit = 0.2 # unit: 1/angstrom, it must be equal to or greater than zero
to_unit = 0.6 # unit: 1/angstrom, it must be smaller than the maximum scattering vector
run_analysis.basic_setup(str_path, from_unit, to_unit, 
                         broadening=0.01, fill_width=0.005) # broadening -> used to simulate diffraction patterns

In [ ]:
plt.close("all")
# Sum of radial variance and average profiles
# profile_type: "mean" or "variance"
# str_name=["structure_name_1", "structure_name_2"]
# The structure names are stored in run_analysis.int_sf.keys()
# "visual_title=True" will show the datetimes of each data
# "title_font_size" -> change the font size of the titles
run_analysis.sum_radial_profile(str_name=[], 
                                profile_type="variance",
                               visual_legend=False,
                               individual_visual=False)

# NMF decomposition

In [ ]:
# Optional process
# NMF - to optimize the number of loading vectors
# rescale_SI=True -> divide each 3D data by its maximum value
# max_normalize=True -> divide every profile by its maximum value
# rescale_0_to_1=True -> rescale every profile from 0 to 1
# Please refer to Scikit-learn, 'nmf' or 'https://github.com/jinseuk56/drca'
# profile_type: "mean" or "variance"
# verbose=True -> it will show the loading vectors and their corresponding coefficient maps
# coeff_map_type: "relative" or "absolute"
# coeff_map_type="absolute" -> the colormap range of all the coefficient maps will be determined from the maximum coefficient to the minimum coefficient

error_list = []
comp_list = []
num_comp_list = np.arange(2, 15, 1)

# Run initial decomposition
run_analysis.NMF_decompose(
    num_comp_list[0],
    profile_type="variance", 
    scale_crop=True, 
    rescale_SI=False, 
    max_normalize=False, 
    rescale_0to1=True, 
    verbose=False, 
    coeff_map_type="relative"
)
error_list.append(run_analysis.run_SI.DR.reconstruction_err_)
comp_list.append(run_analysis.run_SI.DR.components_)

# Iteratively evaluate components
for num_comp in num_comp_list[1:]:
    print(f'Evaluating NMF with components: {num_comp}')
    run_analysis.run_SI.ini_DR(method="nmf", num_comp=num_comp, result_visual=False, intensity_range="relative")
    error_list.append(run_analysis.run_SI.DR.reconstruction_err_)
    comp_list.append(run_analysis.run_SI.DR.components_)

# Plot the errors and slope gradient to identify optimal number of components
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), dpi=100)
ax[0].plot(num_comp_list, error_list, 'k-', marker='*')
ax[0].set_xlabel("Number of loading vectors")
ax[0].set_ylabel("Reconstruction Error")
ax[0].set_title("NMF Reconstruction Error")

slope = np.gradient(error_list)
ax[1].plot(num_comp_list, error_list, 'r-', marker='*')
ax[1].set_xlabel("Number of loading vectors")
ax[1].set_ylabel("Reconstruction Error", color='r')

ax_twin = ax[1].twinx()
ax_twin.plot(num_comp_list, slope, 'b-', marker='o')
ax_twin.set_xticks(num_comp_list)
ax_twin.set_ylabel("Gradient", color='b')
ax_twin.set_title("Error Gradient analysis")

fig.tight_layout()
plt.show()

In [ ]:
plt.close("all")
# NMF - to optimize the number of loading vectors
# rescale_SI=True -> divide each 3D data by its maximum value
# max_normalize=True -> divide every profile by its maximum value
# rescale_0_to_1=True -> rescale every profile from 0 to 1
# scale_crop=True -> the profiles will be cropped in the reciprocal distance range, otherwise in the index range
# Please refer to Scikit-learn, 'nmf' or 'https://github.com/jinseuk56/drca'
# profile_type: "mean" or "variance"
# verbose=True -> it will show the loading vectors and their corresponding coefficient maps
# coeff_map_type: "relative" or "absolute"
# 'relative' -> color scale for each 3D data, 'absolute'-> color scale for all 3D data

num_comp = 9
run_analysis.NMF_decompose(num_comp, 
                           profile_type="variance", 
                           scale_crop=True, 
                           rescale_SI=False, 
                           max_normalize=False, 
                           rescale_0to1=True, 
                           verbose=False,
                           coeff_map_type="relative")

In [ ]:
plt.close("all")
# NMF - loading vectors and their coefficient maps
# If the number of loading vectors exceeds the number of the preset colormaps (currently five),
# it will show the coefficient maps for only 5 loading vectors
# lv_show [loading vector number list] -> you can choose which coefficient map to show
# The colormap list can be accessed run_analysis.cm_rep -> new colormaps can be added
# "visual_title=True" will show the datetimes of each data
# "title_font_size" -> change the font size of the titles
run_analysis.NMF_result(lv_show=None, visual_title=True, title_font_size=10)

In [ ]:
# This below line will show the current preset colormaps
# run_analysis.print_colormaps()

In [ ]:
plt.close("all")
# NMF - show the pixels with high coefficients for each loading vector and the averaged profiles for those pixels
# str_name=["structure_name_1", "structure_name_2"]
# percentile_threshold -> if 90, only the pixels with the 10% highest coefficients remain
# "visual_title=True" will show the datetimes of each data
# "title_font_size" -> change the font size of the titles
# 'visual_individual=False' -> show only the averaged profiles
by_nmf_lv = run_analysis.NMF_comparison(str_name=[], percentile_threshold=90, visual_individual=False)

In [ ]:
# Plot peaks in the averaged profiles generated from the above cell
plt.close("all")
fill_width = 0.001
prominence_profile = 0.0005

fig_lv, ax_lv = plt.subplots(figsize=(7, 4.5), dpi=100)
for l, line in enumerate(by_nmf_lv):
    line = line[run_analysis.range_ind[0]:run_analysis.range_ind[1]].copy()
    peaks = find_peaks(line, prominence=prominence_profile)[0]
    peaks = peaks * run_analysis.pixel_size_inv_Ang
    peaks = peaks + run_analysis.from_
    
    tmp_ax, = ax_lv.plot(run_analysis.x_axis, line, c=run_analysis.color_rep[l+1], label=f'LV {l+1}')
    shadow_effect = path_effects.withStroke(linewidth=3, foreground='gray')
    tmp_ax.set_path_effects([shadow_effect])

    for ip, peak in enumerate(peaks):
        if run_analysis.from_ <= peak <= run_analysis.to_:
            print(f"LV {l+1} Peak: {peak:.4f}")
            ax_lv.axvline(peak, ls=':', lw=1.5, c=run_analysis.color_rep[l+1], alpha=0.5)
            ax_txt = ax_lv.text(peak, np.max(line) * 0.7, f"{peak:.3f}", c=run_analysis.color_rep[l+1], fontsize=7)
            shadow_effect_txt = path_effects.withStroke(linewidth=1, foreground='gray')
            ax_txt.set_path_effects([shadow_effect_txt])           

ax_lv.set_xlabel("1/Å")
ax_lv.set_ylabel("Normalized Intensity")
ax_lv.set_title("Peaks in Average Radial Profiles")
ax_lv.legend(loc='upper right')
fig_lv.tight_layout()
plt.show()

In [ ]:
# Save the mean profiles of the pixels with high coefficients loading vector by loading vector
# import hyperspy.api as hs
# by_nmf_lv = np.asarray(by_nmf_lv)
# print(by_nmf_lv.shape)
# by_nmf_lv = hs.signals.Signal1D(by_nmf_lv)
# by_nmf_lv.axes_manager[0].unit = "loading vector"
# by_nmf_lv.axes_manager[1].scale = run_analysis.pixel_size_inv_Ang
# by_nmf_lv.axes_manager[1].unit = "1/Å"
# by_nmf_lv.save('%s_mean_profiles_loading_vector.hspy'%run_analysis.formatted, overwrite=True)

In [ ]:
# # Compare high-coefficient areas by subfolder and loading vector
# plt.close("all")
# run_analysis.high_coeff_area_comparison()

In [ ]:
# # Generate image-based reports of high-coefficient thresholding for a specific data
# # It will integrate radial profiles and original diffraction patterns, respectively, based on the high-coefficient maps
# # 'prominence_lv' and 'prominence_profile' are used for 'find_peaks' 
# plt.close("all")
# run_analysis.NMF_summary_save_specific(specific_data=[''], save=False, also_dp=True, # specific_data = ['a keyword in the file name']
#                           log_scale_dp=True, also_tiff=False, 
#                           fill_width=0.01, prominence_lv=0.001, 
#                           prominence_profile=0.001)

In [ ]:
# # Generate image-based reports of high-coefficient thresholding for all data
# # It will integrate radial profiles and original diffraction patterns, respectively, based on the high-coefficient maps
# # 'prominence_lv' and 'prominence_profile' are used for 'find_peaks' 
# plt.close("all")
# run_analysis.NMF_summary_save(save=False, also_tiff=False, also_dp=False, 
#                               log_scale_dp=True, fill_width=0.005, 
#                               prominence_lv=0.05, prominence_profile=0.005)

# Clustering of small areas

In [ ]:
plt.close("all")
# Detect small areas in each mask and calculate their centroid and boundary positions
# data_key='a keyword in the file name'
# 'eps' and 'min_sample' are the parameters for DBSCAN clustering algorithm
run_analysis.effective_small_area(data_key='', threshold_map="NMF", eps=4.0, min_sample=20)

In [ ]:
plt.close("all")
# Obtain the sum of 2D diffraction patterns for each small area
# 'virtual_4D=True' -> averaged 2D diffraction patterns of each small area will be stored in 'self.virtual_lv' (only for NMF)
run_analysis.small_area_investigation(save=False, also_tiff=False, virtual_4D=True)

In [ ]:
plt.close("all")
# Check the overlap between small areas by loading vector (if 'threshold_map="NMF" above)
run_analysis.overlap_check()

In [ ]:
plt.close("all")
# Obtain single-phase regions for each data
run_analysis.single_phase_investigation(visual=True, fig_save=False, 
                                        dp_shape=[515, 515], crop_ind=[0, 515, 0, 515],
                                        eps=4.0, min_sample=20,
                                        boundary_method="custom",
                                        fill_method="path")

In [ ]:
# Save single-phase analysis results
save_pkl = {}
save_pkl["boundary_lv_split"] = run_analysis.boundary_lv_split
save_pkl["centroid_lv_split"] = run_analysis.centroid_lv_split
save_pkl["clustered_lv_split"] = run_analysis.clustered_lv_split
save_pkl["pos_lv_pixel_split"] = run_analysis.pos_lv_pixel_split
save_pkl["num_lv_pixel_split"] = run_analysis.num_lv_pixel_split

import pickle
with open('%s_single_phase.pkl'%run_analysis.formatted, 'wb') as f:
    pickle.dump(save_pkl, f)

In [ ]:
# Save the mean profiles of the pixels with high coefficients loading vector by loading vector
mean_rvp_by_lv = []
for i in range(len(run_analysis.subfolders)):
    for lv in range(run_analysis.num_comp):
        mean_rvp_by_lv.append(run_analysis.mean_rvp['sub_index_%d_LV%d'%(i+1, lv+1)]/run_analysis.num_pixel['sub_index_%d_LV%d'%(i+1, lv+1)])
mean_rvp_by_lv = np.asarray(mean_rvp_by_lv)
print(mean_rvp_by_lv.shape)

by_nmf_lv_save = hs.signals.Signal1D(mean_rvp_by_lv)
by_nmf_lv_save.axes_manager[0].units = "%d"%run_analysis.num_comp
by_nmf_lv_save.axes_manager[0].name = "%d"%len(run_analysis.subfolders)
by_nmf_lv_save.axes_manager[1].scale = run_analysis.pixel_size_inv_Ang
by_nmf_lv_save.axes_manager[1].unit = "1/Å"
by_nmf_lv_save.save('%s_mean_rvp_LV_clustering.hspy'%run_analysis.formatted, overwrite=True)

In [ ]:
k = 0
for i in range(len(run_analysis.subfolders)):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=100)
    total_mean = np.mean(run_analysis.radial_var_sum_split[i], axis=0)
    ax.plot(run_analysis.x_axis, total_mean[run_analysis.range_ind[0]:run_analysis.range_ind[1]], 'k:')
    ax_twin = ax.twinx()
    for lv in range(run_analysis.num_comp):
        line = mean_rvp_by_lv[lv+k]
        tmp_ax, = ax_twin.plot(run_analysis.x_axis, 
               line[run_analysis.range_ind[0]:run_analysis.range_ind[1]], c=run_analysis.color_rep[lv+1], label='sub index %d lv %d'%(i+1, lv+1))
        
        shadow_effect = path_effects.withStroke(linewidth=3, foreground='gray')
        tmp_ax.set_path_effects([shadow_effect])

    k += run_analysis.num_comp

    ax_twin.legend(fontsize=10)
    fig.tight_layout()
    plt.show()

In [ ]:
# Save the mean profiles of the pixels with high coefficients loading vector by loading vector
mean_rmp_by_lv = []
for i in range(len(run_analysis.subfolders)):
    for lv in range(run_analysis.num_comp):
        mean_rmp_by_lv.append(run_analysis.mean_rmp['sub_index_%d_LV%d'%(i+1, lv+1)]/run_analysis.num_pixel['sub_index_%d_LV%d'%(i+1, lv+1)])
mean_rmp_by_lv = np.asarray(mean_rmp_by_lv)
print(mean_rmp_by_lv.shape)

by_nmf_lv_save = hs.signals.Signal1D(mean_rmp_by_lv)
by_nmf_lv_save.axes_manager[0].units = "%d"%run_analysis.num_comp
by_nmf_lv_save.axes_manager[0].name = "%d"%len(run_analysis.subfolders)
by_nmf_lv_save.axes_manager[1].scale = run_analysis.pixel_size_inv_Ang
by_nmf_lv_save.axes_manager[1].units = "1/Å"
by_nmf_lv_save.save('%s_mean_rmp_LV_clustering.hspy'%run_analysis.formatted, overwrite=True)

In [ ]:
k = 0
for i in range(len(run_analysis.subfolders)):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=100)
    total_mean = np.mean(run_analysis.radial_var_sum_split[i], axis=0)
    ax.plot(run_analysis.x_axis, total_mean[run_analysis.range_ind[0]:run_analysis.range_ind[1]], 'k:')
    ax_twin = ax.twinx()
    for lv in range(run_analysis.num_comp):
        line = mean_rmp_by_lv[lv+k]
        tmp_ax, = ax_twin.plot(run_analysis.x_axis, 
               line[run_analysis.range_ind[0]:run_analysis.range_ind[1]], c=run_analysis.color_rep[lv+1], label='sub index %d lv %d'%(i+1, lv+1))

        shadow_effect = path_effects.withStroke(linewidth=3, foreground='gray')
        tmp_ax.set_path_effects([shadow_effect])
    
    k += run_analysis.num_comp
    ax_twin.legend(fontsize=10)
    fig.tight_layout()
    plt.show()

In [ ]:
import gc
# Save the averaged diffraction patterns of single-phase areas
for i in range(len(run_analysis.subfolders)):
    for lv in range(run_analysis.num_comp):
        dp_storage = np.asarray(run_analysis.dp_storage['sub_index_%d_LV%d'%(i+1, lv+1)])
        dp_storage = dp_storage.astype(np.uint8)
        hs_save = hs.signals.Signal2D(dp_storage)
        del dp_storage
        print(hs_save)
        hs_save.axes_manager[0].units = "%d"%run_analysis.num_comp
        hs_save.axes_manager[0].name = "%d"%len(run_analysis.subfolders)
        hs_save.axes_manager[1].scale = run_analysis.pixel_size_inv_Ang
        hs_save.axes_manager[1].units = "1/Å"
        hs_save.axes_manager[2].scale = run_analysis.pixel_size_inv_Ang
        hs_save.axes_manager[2].units = "1/Å"
        hs_save.save("%s_2D_DP_sub_index_%d_LV%d_clustering.hspy"%(run_analysis.formatted, i+1, lv+1), overwrite=True)
        del hs_save
        gc.collect()

In [ ]:
# Generate the summary report (includes NMF parameter details and phase area occupancy tables)
rpa_report = run_analysis.summary_report()

# Print the report directly inside the notebook (in markdown format)
from IPython.display import display, Markdown
display(Markdown(rpa_report))

# Save the report as a markdown file
with open("radial_profile_analysis_report.md", "w", encoding="utf-8") as f:
    f.write(rpa_report)

# Save the object state to a pickle file for scientific cross-analysis
run_analysis.save_state("rpa_state.pkl")

# High variances

In [ ]:
plt.close("all")
# Peak detection
# Please refer to SciPy 'find_peaks' for details
# scattering vector range -> [peak_position-half_width, peak_position+half_width]
half_width = 0.005
run_analysis.scattering_range_of_interest(fill_width=half_width,
                                         profile_type="variance",
                                         prominence=0.01,
                                         height=None,
                                         width=None,
                                         distance=None,
                                         threshold=None)

In [ ]:
plt.close("all")
# Variance maps for the specified scattering vector range
# Average and standard deviation of variances for the specified scattering vector range
# Threshold maps - the pixels with high variances will be 1, otherwise 0
### abs_threshold - > absolute threshold value
### percentile_threshold -> if 90, only the pixels with the 10% highest variances remain

# This will show the results for each peak detected in the cell above
# The loop is not necessary
sum_radial_list = []
for i, peak in enumerate(run_analysis.peak_sub[run_analysis.subfolders[0]]):
    peak_selected = peak
    print(run_analysis.subfolders[0], ", peak No. %d - range: %.3f ~ %.3f"%(i+1, peak-half_width, peak+half_width))
    run_analysis.variance_map(sv_range=[peak_selected-half_width, peak_selected+half_width])
    tmp_sum_radial = run_analysis.high_variance_map(percentile_threshold=90)
    print("threshold value to determine the high variances: %.3f"%run_analysis.abs_threshold)
    sum_radial_list.append(tmp_sum_radial)

    # optional - if you want to see the mean 2D diffraction patterns for individual data
    # It will take a long time due to loading each 4D data
    # run_analysis.summary_save(save=False, 
    #                           also_dp=True,
    #                           log_scale_dp=True)

In [ ]:
# Save the mean profiles of the pixels with high variances peak by peak
# import hyperspy.api as hs
# sum_radial_list = np.asarray(sum_radial_list)
# print(sum_radial_list.shape)
# sum_radial_list = hs.signals.Signal1D(sum_radial_list)
# sum_radial_list.axes_manager[0].unit = "peak"
# sum_radial_list.axes_manager[1].scale = run_analysis.pixel_size_inv_Ang
# sum_radial_list.axes_manager[1].unit = "1/Å"
# sum_radial_list.save('%s_mean_profiles_by_peak.hspy'%run_analysis.formatted, overwrite=True)

In [ ]:
plt.close("all")
# Compare the mean of radial profiles
# run_analysis.basic_setup(str_path, 0.1, 1.0, broadening=0.01) # specify the scattering vector range
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

# sel_peak_num = np.array([1, 2, 3]) - 1 # select a few peaks of interest
sel_peak_num = np.arange(len(run_analysis.peak_sub[run_analysis.subfolders[0]])) # for all the peaks

for i in sel_peak_num:
    ax.plot(run_analysis.x_axis, 
               sum_radial_list[i][int(run_analysis.from_/run_analysis.pixel_size_inv_Ang):int(run_analysis.to_/run_analysis.pixel_size_inv_Ang)], 
               c=run_analysis.color_rep[i], label="peak No.%d"%(i+1))
ax.legend()
ax.set_xlabel("Scattering vector (1/Å)")
ax.set_facecolor("lightgray")
fig.tight_layout()
plt.show()